# 02 — Data Cleaning

Ce notebook applique les règles de nettoyage établies dans `01_data_understanding.ipynb`. Chaque étape rappelle la décision et sa justification avant le code.

## 1. Chargement des données brutes

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/logements-sociaux-finances-a-paris.csv", sep=';')
print("Shape initiale :", df.shape)
df.head()


## 2. Suppression des colonnes sans valeur analytique

- `commentaires` : 100% de valeurs manquantes (0 valeur non-nulle) → aucune information.
- `ville` : constante ("Paris" partout) → aucune valeur discriminante.


In [ ]:
cols_to_drop = ['commentaires', 'ville']
df = df.drop(columns=cols_to_drop)
print(f"Colonnes supprimées : {cols_to_drop}")
print("Shape après suppression :", df.shape)


## 3. Exclusion des lignes avec valeurs négatives

2 lignes concernées (`id_livraison` 2000587 et 2000900). Le total (`nb_logmt_total`) est cohérent
avec la somme des sous-catégories dans les deux cas → probablement des corrections/annulations
d'une déclaration antérieure, pas des lignes de logements réelles. Impact négligeable (2/4174 lignes).


In [ ]:
neg_cols = ['nb_logmt_total', 'nb_plai', 'nb_plus', 'nb_pluscd', 'nb_pls']
neg_mask = (df[neg_cols] < 0).any(axis=1)

print(f"Lignes exclues : {neg_mask.sum()}")
print(df.loc[neg_mask, ['id_livraison', 'adresse_programme', 'annee'] + neg_cols])

df = df.loc[~neg_mask].copy()
print("\nShape après exclusion :", df.shape)


## 4. `id_livraison` dupliqués — conservés tels quels

Les 32 IDs dupliqués (65 lignes) représentent des tranches de financement différentes ou des phases
de livraison distinctes d'un même programme — **ce ne sont pas des doublons**. `id_livraison` identifie
un programme, pas une ligne unique. Aucune suppression, mais on documente le comportement pour ne pas
utiliser `id_livraison` comme clé unique plus tard dans l'analyse.


In [ ]:
n_dupe_ids = df['id_livraison'].duplicated(keep=False).sum()
print(f"Lignes concernées par un id_livraison dupliqué (conservées) : {n_dupe_ids}")
print("Rappel : id_livraison n'est PAS une clé unique par ligne dans ce dataset.")


## 5. `bs` = "(vide)" → converti en vraie valeur manquante (NaN)

C'est un placeholder texte pour une valeur manquante, pas une vraie catégorie. Le convertir en `NaN`
évite qu'il soit compté comme une modalité à part entière dans de futures agrégations (`value_counts`,
`groupby`, etc.), même si la colonne n'est pas utilisée dans l'hypothèse principale.


In [ ]:
n_vide = (df['bs'] == '(vide)').sum()
df['bs'] = df['bs'].replace('(vide)', np.nan)
print(f"'(vide)' remplacé par NaN sur {n_vide} lignes")


## 6. Correction des types de données

- `annee` : déjà en `int64`, OK.
- `mode_real`, `nature_programme` : conversion en `category` pour optimiser la mémoire et
  rendre les valeurs autorisées explicites.


In [ ]:
cat_cols = ['mode_real', 'nature_programme']
for c in cat_cols:
    df[c] = df[c].astype('category')


df.dtypes


## 7. Vérifications finales


In [ ]:
print("Shape finale :", df.shape)
print("\nValeurs manquantes restantes :")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nValeurs négatives restantes :")
print((df[neg_cols] < 0).sum().sum())
print("\nLignes dupliquées exactes :", df.duplicated().sum())

## 8. Sauvegarde du dataset nettoyé

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/logements-sociaux-paris-clean.csv", index=False, sep=';')
print("Sauvegardé :", df.shape)
